# From spreadsheets to defensible maps

**Atlas source reading copy · browser-network**

[Course guide](../index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-Sea-Level-Rise/blob/66065673b47f9bc5c0f6e5009ac1c924a9f5200c/content/notebooks/04_data_and_geography.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability.

**Provenance:** `content/notebooks/04_data_and_geography.ipynb` at `66065673b47f9bc5c0f6e5009ac1c924a9f5200c`; original SHA-256 `d6c0a26507630c72850aa266b20b81424c35126468862aaffa5ffd2f4f82affb`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# From spreadsheets to defensible maps

**Time:** 45 minutes · **Prerequisites:** basic Python; run cells from top to bottom.

Reshape wide Power BI fields into a tidy panel; validate joins; explain why map scale, coordinate systems and privacy matter.

[Run this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/04_data_and_geography.ipynb) · [Full-screen dashboard](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/dashboard/) · [Official poster](https://plus.iasociety.org/sites/default/files/2024-09/e-poster_755.pdf)

Each notebook runs independently. Use **Run → Run All Cells**; the first package download can take a minute. Save a downloaded copy to keep your work. The read-only book includes executed examples.

In [ ]:
# First run downloads Python packages in JupyterLite; subsequent runs use browser caches.
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'plotly', 'ipywidgets'])
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
from coastlab import (DATA, SCENARIOS, YEARS, COLORS, load_data, select, cards,
                      toy_slr, toy_probability, cumulative_probability,
                      haversine_km, show, dashboard, land_traces)
facilities, exposure = load_data()
published = pd.read_csv(DATA / 'poster_aggregates.csv')
print('Ready: public poster aggregates + clearly labeled fictional facility fixtures.')

## Inspect the grain before the map

The Power BI model is wide: A/B/C columns encode RCP 2.6/4.5/8.5 and suffixes 20–90 and 00 encode decades 2020–2090 and 2100. The confirmed tab titles establish the scenario mapping. Names such as R26EAE400 and R65EAE50 need explicit review rather than silent renaming.

Our fictional panel uses one row per **facility × scenario × year**. Facility attributes live in a separate dimension table. Never sum the dimension's service load across all scenario-year rows: each facility would be repeated 27 times.

In [ ]:
display(facilities.head(3))
display(exposure.head(6))
joined=facilities.merge(exposure,on='facility_id',validate='one_to_many')
print('Facilities:',len(facilities),'Panel rows:',len(joined))
print('Naive repeated sum / true service-load sum:',joined.service_load.sum()/facilities.service_load.sum())
assert len(joined)==len(facilities)*3*9

In [ ]:
# Demonstrate a wide-to-long conversion using FICTIONAL category columns.
wide=pd.DataFrame({'facility_id':['DEMO-X','DEMO-Y'],'A30':['YELLOW','NOFILL'],'B30':['ORANGE','NOFILL'],'C00':['RED','YELLOW']})
long=wide.melt(id_vars='facility_id',var_name='source_field',value_name='source_category')
long['scenario']=long.source_field.str[0].map({'A':'RCP 2.6','B':'RCP 4.5','C':'RCP 8.5'})
long['year']=long.source_field.str[1:].map(lambda suffix:2100 if suffix=='00' else 2000+int(suffix))
display(long)

## Coordinates are not interchangeable measurements

Longitude/latitude in EPSG:4326 are angular coordinates. Distances in degrees are not distances in metres. A vertical datum is a different concept again: elevation relative to MHHW cannot be mixed casually with ellipsoidal or orthometric elevation.

Natural Earth's 1:110 million land outlines are sufficient for a global orientation map. They do not resolve local drainage, seawalls or a building footprint. The map below plots **invented** points; points in water or implausible locations are expected because they are random teaching fixtures, not a land-filtered facility inventory.

In [ ]:
rows=select(facilities,exposure,country='Thailand',year=2030)
fig=go.Figure(land_traces())
fig.add_trace(go.Scatter(x=rows.longitude,y=rows.latitude,mode='markers',text=rows.facility_id,
    marker=dict(size=9,color=rows.annual_probability,cmin=0,cmax=1,colorscale='YlOrRd'),name='Fictional sites'))
fig.update_layout(title='Fictional Thai-city examples; coarse coastline for context',xaxis=dict(title='Longitude',range=[99,102]),yaxis=dict(title='Latitude',range=[12,15]))
show(fig)
a,b=rows.iloc[0],rows.iloc[1]
print('Great-circle distance, not road travel distance (km):',round(haversine_km(a.longitude,a.latitude,b.longitude,b.latitude),2))

## Lab: catch a misleading join

Duplicate a facility identifier in the dimension table and try a `validate='one_to_many'` merge. Catch the resulting error, then explain why removing duplicates without knowing which record is authoritative is also unsafe.

**Worked check:** A valid panel has 192 unique fictional facilities, 3 scenarios, 9 years and 5,184 rows. Selecting one year and one scenario returns at most 192 rows. A city filter changes the cohort before aggregation.

**Privacy checkpoint:** Removing names does not anonymize precise coordinates. This public repository contains only poster aggregates and independently generated fictional facility data. A private audit workflow should stay in an approved environment and should never commit raw workbooks or UID-coordinate pairs.

**Deliverable:** A one-page data contract specifying grain, join keys, coordinate reference system, missing-value treatment and allowed publication level. Consult the book's data dictionary and Power BI crosswalk.